In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("Water Quality.csv")
TARGET_PARAMETER = "Specific conductance"
pd.set_option("display.max_columns", 50)


In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Place 'Water Quality.csv' beside this notebook, or update DATA_PATH.")
df = pd.read_csv(DATA_PATH, low_memory=False)
required = {"MonitoringLocationID", "ActivityStartDate", "CharacteristicName", "ResultValue"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
df["CharacteristicName"] = df["CharacteristicName"].astype("string").str.strip()
df["MonitoringLocationID"] = df["MonitoringLocationID"].astype("string").str.strip()
df = df.dropna(subset=["MonitoringLocationID", "ActivityStartDate", "CharacteristicName", "ResultValue"]).copy()
df["SampleDate"] = df["ActivityStartDate"].dt.floor("D")

if not (df["CharacteristicName"].str.casefold() == TARGET_PARAMETER.casefold()).any():
    matches = sorted(x for x in df["CharacteristicName"].dropna().unique() if "conduct" in x.casefold())
    raise ValueError(f"Target {TARGET_PARAMETER!r} not found. Similar names: {matches[:20]}")

wide = (df.groupby(["MonitoringLocationID", "SampleDate", "CharacteristicName"])["ResultValue"]
          .median().unstack("CharacteristicName").reset_index())
target_col = next((c for c in wide.columns if isinstance(c, str) and c.casefold() == TARGET_PARAMETER.casefold()), None)
if target_col is None:
    raise ValueError("Target column was not created after pivoting.")

wide = wide.sort_values(["SampleDate", "MonitoringLocationID"]).reset_index(drop=True)
numeric_cols = wide.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in numeric_cols if c != target_col and wide[c].notna().sum() >= max(30, int(0.05 * len(wide)))]
data = wide.dropna(subset=[target_col]).sort_values("SampleDate").reset_index(drop=True)
features = [c for c in features if c in data.columns]

if len(data) < 50:
    raise ValueError(f"Only {len(data)} usable target rows. This workflow recommends at least 50.")
if not features:
    raise ValueError("No other measured parameters have enough matched records to use as predictors.")

split = min(max(int(0.8 * len(data)), 1), len(data) - 1)
train, test = data.iloc[:split].copy(), data.iloc[split:].copy()
X_train, X_test = train[features], test[features]
y_train, y_test = train[target_col], test[target_col]
print("Rows:", len(data), "| predictors:", len(features), "| train:", len(train), "| test:", len(test))
print("Train dates:", train.SampleDate.min(), "to", train.SampleDate.max())
print("Test dates:", test.SampleDate.min(), "to", test.SampleDate.max())
print("Predictors:", features)


Rows: 7222 | predictors: 50 | train: 5777 | test: 1445
Train dates: 2000-01-03 00:00:00 to 2017-09-12 00:00:00
Test dates: 2017-09-14 00:00:00 to 2023-12-06 00:00:00
Predictors: ['Alkalinity, total', 'Aluminum', 'Antimony', 'Apparent color', 'Arsenic', 'Barium', 'Beryllium', 'Bismuth', 'Boron', 'Bromine', 'Cadmium', 'Calcium', 'Chloride', 'Chromium', 'Cobalt', 'Copper', 'Dissolved oxygen (DO)', 'Dissolved oxygen saturation', 'Fluoride', 'Gallium', 'Gran acid neutralizing capacity', 'Inorganic carbon', 'Iron', 'Lanthanum', 'Lead', 'Lithium', 'Magnesium', 'Manganese', 'Molybdenum', 'Nickel', 'Nitrate', 'Organic carbon', 'Potassium', 'Rubidium', 'Selenium', 'Silver', 'Sodium', 'Strontium', 'Sulfate', 'Temperature, water', 'Thallium', 'Tin', 'Titanium', 'Total Nitrogen, mixed forms', 'Total Phosphorus, mixed forms', 'Turbidity', 'Uranium', 'Vanadium', 'Zinc', 'pH']


In [4]:
try:
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Install packages using %pip install xgboost lightgbm, then restart the kernel.") from exc

print("Boosting packages imported successfully.")


Boosting packages imported successfully.


In [5]:
n_splits = min(5, max(2, len(train) // 30))
cv = TimeSeriesSplit(n_splits=n_splits)

xgb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE,
                           n_jobs=1, tree_method="hist"))
])
xgb_params = {
    "model__n_estimators": [200, 400, 700],
    "model__max_depth": [2, 3, 5, 7],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__reg_lambda": [1, 3, 8]
}
lgb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LGBMRegressor(objective="regression", random_state=RANDOM_STATE,
                            n_jobs=1, verbosity=-1))
])
lgb_params = {
    "model__n_estimators": [200, 400, 700],
    "model__max_depth": [-1, 3, 5, 7],
    "model__num_leaves": [7, 15, 31, 63],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__reg_lambda": [0, 1, 5]
}

xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=15,
    scoring="neg_root_mean_squared_error", cv=cv, random_state=RANDOM_STATE,
    n_jobs=-1, verbose=1, refit=True)
lgb_search = RandomizedSearchCV(lgb_pipe, lgb_params, n_iter=15,
    scoring="neg_root_mean_squared_error", cv=cv, random_state=RANDOM_STATE,
    n_jobs=-1, verbose=1, refit=True)

xgb_search.fit(X_train, y_train)
lgb_search.fit(X_train, y_train)
print("XGBoost best CV RMSE:", round(-xgb_search.best_score_, 4))
print("XGBoost best parameters:", xgb_search.best_params_)
print("LightGBM best CV RMSE:", round(-lgb_search.best_score_, 4))
print("LightGBM best parameters:", lgb_search.best_params_)


Fitting 5 folds for each of 15 candidates, totalling 75 fits
Fitting 5 folds for each of 15 candidates, totalling 75 fits
XGBoost best CV RMSE: 43.1201
XGBoost best parameters: {'model__subsample': 1.0, 'model__reg_lambda': 3, 'model__n_estimators': 700, 'model__min_child_weight': 3, 'model__max_depth': 5, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.7}
LightGBM best CV RMSE: 47.2391
LightGBM best parameters: {'model__subsample': 1.0, 'model__reg_lambda': 0, 'model__num_leaves': 63, 'model__n_estimators': 400, 'model__max_depth': 7, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.7}
